<!-- FULL: keep, DEMO: keep -->
<div style='display:flex; align-items:center; justify-content:space-between; border-bottom:3px solid rgb(255,106,0); padding-bottom:1em;'>
<div>
<span style='color:rgb(22,60,105); font-size:1.8em; font-weight:bold;'>Introduction to Deep Learning</span><br>
<span style='color:rgb(0,85,100); font-size:1.3em;'>Session 6 &mdash; 3: nn.Module &amp; Going Deeper</span><br>
<span style='color:rgb(0,85,100); font-size:1.0em;'>Magda Gregorová &nbsp;·&nbsp; THWS &nbsp;·&nbsp; May 2026</span>
</div>
<img src='../../Common/Pics/thws-logo_vert_en_orange-rgb.png' style='height:80px;'/>
</div>

<!-- FULL: keep, DEMO: delete -->
*We have tensors and autograd. Next: organise parameters and layers into a network, then make it deeper.*

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import matplotlib.pyplot as plt
import sys
sys.path.append('../../A2/')
from helpers import load_data


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Why <code>nn.Module</code>?
</div>

<!-- FULL: keep, DEMO: delete -->
As networks grow deeper, managing raw parameter tensors becomes error-prone:
tracking all weights and biases, passing them to the optimiser, switching train/eval behaviour.

`nn.Module` handles parameter registration, device placement, and train/eval switching automatically.

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(179,27,27); background:rgb(255,240,240); color:rgb(179,27,27); font-size:1.1em; font-weight:bold;'>
  &#9888; Failure case &mdash; forgetting super().__init__()
</div>

<!-- FULL: keep, DEMO: delete -->
Every `nn.Module` subclass must call `super().__init__()` as the first line of `__init__`. Without it, PyTorch's parameter registration machinery is not set up and you get a cryptic `AttributeError`.

In [ ]:
class BrokenModel(nn.Module):
    def __init__(self):
        # missing: super().__init__()
        self.linear = nn.Linear(4, 1)   # AttributeError!

BrokenModel()

<!-- FULL: keep, DEMO: shorten -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Custom <code>nn.Module</code> — minimal example
</div>

<!-- FULL: keep, DEMO: delete -->
Subclass `nn.Module`, implement `__init__` (register layers) and `forward` (computation). Nothing else required.

In [ ]:
import torch.nn as nn

class LinearModel(nn.Module):
    """Single linear layer: y = X @ theta_1.T + theta_0."""

    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.linear(x)


model = LinearModel(in_features=4, out_features=1)
print(model)

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>nn.Linear</code> and parameters
</div>

<!-- FULL: keep, DEMO: delete -->
`nn.Linear(in, out)` stores `weight` (shape `[out, in]`) and `bias` (shape `[out]`) as `nn.Parameter` objects — automatically tracked by `model.parameters()`.

Correspondence with our notation: `weight` $\leftrightarrow$ $\Theta_1$, `bias` $\leftrightarrow$ $\Theta_0$.

In [ ]:
# inspect parameters
for name, param in model.named_parameters():
    print(f'{name:30s}  shape={str(param.shape):15s}  requires_grad={param.requires_grad}')

In [ ]:
# correspondence with our manual notation:
# nn.Linear stores:  weight  shape (out, in)  <->  theta_1
#                    bias    shape (out,)      <->  theta_0
print(f'weight (theta_1): {model.linear.weight.shape}')
print(f'bias   (theta_0): {model.linear.bias.shape}')

<!-- FULL: keep, DEMO: shorten -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  MLP — <code>nn.Module</code> with multiple layers
</div>

In [ ]:
class MLP(nn.Module):
    """Two-layer MLP: Linear -> ReLU -> Linear."""

    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        self.layer1 = nn.Linear(in_features, hidden)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(hidden, out_features)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x


mlp = MLP(in_features=4, hidden=8, out_features=1)
print(mlp)
print(f'\nTotal parameters: {sum(p.numel() for p in mlp.parameters())}')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(0,85,100); background:rgb(235,245,247); color:rgb(0,85,100); font-size:1.1em; font-weight:bold;'>
  &#8644; Alternatives &mdash; defining an MLP — three styles
</div>

<!-- FULL: keep, DEMO: delete -->
All three produce equivalent networks. Choice depends on flexibility needed:

- **Explicit attributes** — most readable, easiest to debug
- **`nn.ModuleList`** — useful when number of layers varies at runtime
- **`nn.Sequential`** — most concise for simple feed-forward networks

In [ ]:
# Style 1 — explicit named attributes
class MLP_v1(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 8)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(8, 1)
    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))

# Style 2 — nn.ModuleList with loop
class MLP_v2(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1)])
    def forward(self, x):
        for layer in self.layers: x = layer(x)
        return x

# Style 3 — nn.Sequential
mlp_v3 = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))

# All have the same parameter count
for name, m in [('v1', MLP_v1()), ('v2', MLP_v2()), ('v3', mlp_v3)]:
    print(f'{name}: {sum(p.numel() for p in m.parameters())} parameters')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>nn.Sequential</code> shortcut
</div>

<!-- FULL: keep, DEMO: delete -->
For simple feed-forward networks, `nn.Sequential` avoids subclassing — list layers in order.

In [ ]:
mlp_seq = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
)
print(mlp_seq)

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(179,27,27); background:rgb(255,240,240); color:rgb(179,27,27); font-size:1.1em; font-weight:bold;'>
  &#9888; Failure case &mdash; using a plain list instead of nn.ModuleList
</div>

<!-- FULL: keep, DEMO: delete -->
Layers stored in a plain Python list are **not** registered as submodules. `model.parameters()` will not see them — they will not be updated during training.

In [ ]:
class BrokenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # plain list — parameters not registered!
        self.layers = [nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1)]

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

broken = BrokenMLP()
print('Parameters found:', list(broken.parameters()))   # empty!

# Fix: use nn.ModuleList
class FixedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1)])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

fixed = FixedMLP()
print('Parameters found:', sum(p.numel() for p in fixed.parameters()))  # correct

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>model.train()</code> and <code>model.eval()</code>
</div>

<!-- FULL: keep, DEMO: delete -->
Some layers (Dropout, BatchNorm — Session 7) behave differently in train vs eval.
Always call `model.eval()` before validation and `model.train()` before resuming training.

In [ ]:
print(f'Default:      training={mlp.training}')
mlp.eval()
print(f'After eval(): training={mlp.training}')
mlp.train()
print(f'After train(): training={mlp.training}')

<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='margin:3em 0 1.5em 0; padding:0.8em 1.2em; background:rgb(22,60,105); border-radius:6px; display:flex; align-items:center; gap:1em;'>
  <span style='color:rgb(255,106,0); font-size:2em; font-weight:bold; line-height:1;'></span>
  <span style='color:white; font-size:1.4em; font-weight:bold;'>Going Deeper</span>
</div>

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Configurable <code>DeepMLP</code>
</div>

<!-- FULL: keep, DEMO: delete -->
With `nn.Module` subclassing, depth and width become constructor arguments.
`nn.Sequential(*layers)` assembles any list of layers dynamically.

In [ ]:
class DeepMLP(nn.Module):
    """MLP with configurable depth and width."""

    def __init__(self, in_features, hidden_sizes, out_features):
        """
        Args:
            in_features:  number of input features
            hidden_sizes: list of hidden layer widths, e.g. [32, 16, 8]
            out_features: number of outputs
        """
        super().__init__()
        layers = []
        sizes  = [in_features] + hidden_sizes
        for i in range(len(sizes) - 1):
            layers.append(nn.Linear(sizes[i], sizes[i+1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(sizes[-1], out_features))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# shallow
shallow = DeepMLP(4, [8], 1)
# deep
deep    = DeepMLP(4, [32, 16, 8], 1)

print('Shallow:', sum(p.numel() for p in shallow.parameters()), 'parameters')
print(shallow)
print('\nDeep:   ', sum(p.numel() for p in deep.parameters()), 'parameters')
print(deep)

<!-- FULL: keep, DEMO: shorten -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Shallow vs deep — convergence comparison
</div>

In [ ]:
def train_model(model, train_loader, val_loader, n_epochs=100, lr=1e-3):
    """Train a model and return train/val loss histories."""
    optimizer    = optim.Adam(model.parameters(), lr=lr)
    loss_fn      = nn.MSELoss()
    train_losses = []
    val_losses   = []
    for _ in range(n_epochs):
        train_losses.append(train_epoch(model, train_loader, loss_fn, optimizer))
        val_losses.append(val_epoch(model, val_loader, loss_fn))
    return train_losses, val_losses


torch.manual_seed(0)
shallow = DeepMLP(4, [8], 1)
tl_s, vl_s = train_model(shallow, train_loader, val_loader)

torch.manual_seed(0)
deep = DeepMLP(4, [32, 16, 8], 1)
tl_d, vl_d = train_model(deep, train_loader, val_loader)

print(f'Shallow — final val loss: {vl_s[-1]:.4f}')
print(f'Deep    — final val loss: {vl_d[-1]:.4f}')

In [ ]:
from plotting import plot_losses_compare
plot_losses_compare(
    [(tl_s, vl_s, 'shallow [8]'),
     (tl_d, vl_d, 'deep [32,16,8]')]
)

<!-- FULL: keep, DEMO: keep -->
<br/>